# LM Alignment — полный пайплайн
**Модель:** Qwen/Qwen3-4B-Instruct-2507 · **GPU:** T4 · **seed:** 42

| # | Метод | Данные | Метрика |
|---|---|---|---|
| 1 | SFT | kid_adult | P_simple |
| 2 | DPO style | kid_adult | P_simple |
| 3 | Reward Model | good_bad | accuracy |
| 4 | DPO quality | good_bad | gold_rm score |
| 5 | SimPO | good_bad | gold_rm score |

**Перед запуском:** загрузи папку `ml-olympiad-red-task/` в корень Google Drive.

In [ ]:
!pip uninstall -y trl 2>/dev/null
!pip install -q "trl>=0.15.0" peft bitsandbytes accelerate datasets safetensors scikit-learn scipy
!pip install -q "torchao>=0.16.0"

import sys
for m in list(sys.modules):
    if m == "trl" or m.startswith("trl."):
        del sys.modules[m]
import trl
print("trl", trl.__version__)

def _check_cpo():
    try:
        from trl import CPOTrainer
        return "trl"
    except ImportError:
        pass
    try:
        from trl.experimental.cpo import CPOTrainer
        return "trl.experimental.cpo"
    except ImportError:
        pass
    from trl.trainer.cpo_trainer import CPOTrainer
    return "trl.trainer.cpo_trainer"

print("CPOTrainer ok:", _check_cpo())


In [ ]:
import os, gc, re, json, random, pickle
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np
import torch
import torch.nn as nn
from scipy.sparse import hstack as sp_hstack

SEED = 42
MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

METRICS = {}


In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

DRIVE_BASE   = '/content/drive/MyDrive/ml-olympiad-red-task'
LOCAL_BASE   = '/content/task'
DATA_DIR     = f'{LOCAL_BASE}/data'
METRICS_DIR  = f'{LOCAL_BASE}/metrics'
GOLD_RM_PATH = f'{METRICS_DIR}/gold_rm'

os.makedirs(LOCAL_BASE, exist_ok=True)
for sub in ['data', 'metrics']:
    dst = f'{LOCAL_BASE}/{sub}'
    if not os.path.exists(dst):
        shutil.copytree(f'{DRIVE_BASE}/{sub}', dst)
        print(f'Copied {sub}/')
    else:
        print(f'Already exists: {sub}/')

print('data:', os.listdir(DATA_DIR))
print('metrics:', os.listdir(METRICS_DIR))


In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

kid_adult_raw    = load_jsonl(f'{DATA_DIR}/kid_adult.jsonl')
good_bad_raw     = load_jsonl(f'{DATA_DIR}/good_bad.jsonl')
test_style_raw   = load_jsonl(f'{DATA_DIR}/public_test_style.jsonl')
test_quality_raw = load_jsonl(f'{DATA_DIR}/public_test_quality.jsonl')

print(f'kid_adult    : {len(kid_adult_raw):>5}  keys={list(kid_adult_raw[0].keys())}')
print(f'good_bad     : {len(good_bad_raw):>5}  keys={list(good_bad_raw[0].keys())}')
print(f'test_style   : {len(test_style_raw):>5}')
print(f'test_quality : {len(test_quality_raw):>5}')

TEST_STYLE_PROMPTS    = [d['prompt']    for d in test_style_raw]
TEST_QUALITY_PROMPTS  = [d['prompt']    for d in test_quality_raw]
TEST_QUALITY_CHOSEN   = [d['chosen']    for d in test_quality_raw]
TEST_QUALITY_REJECTED = [d['rejected']  for d in test_quality_raw]


In [ ]:
# style_clf.pkl = dict{'clf': LogisticRegression, 'vecs': (TfidfVec, TfidfVec)}
with open(f'{METRICS_DIR}/style_clf.pkl', 'rb') as f:
    _bundle = pickle.load(f)

print('Bundle type:', type(_bundle).__name__)

if isinstance(_bundle, dict):
    _lr   = _bundle['clf']
    _vecs = _bundle['vecs']
    print('clf:', type(_lr).__name__)
    print('vecs:', [type(v).__name__ for v in _vecs])
    def _proba(texts):
        X = sp_hstack([v.transform(texts) for v in _vecs])
        return _lr.predict_proba(X)
else:
    _proba = _bundle.predict_proba

_t = [
    'Это простой ответ для детей с понятными аналогиями.',
    'Данный феномен обусловлен многоуровневым взаимодействием сложных когнитивных процессов.'
]
_p = _proba(_t)
SIMPLE_IDX = int(_p[0][1] > _p[0][0])
print(f'SIMPLE_IDX={SIMPLE_IDX}  simple={_p[0][SIMPLE_IDX]:.3f}  complex={_p[1][SIMPLE_IDX]:.3f}')

def compute_p_simple(texts):
    safe  = [t if t and t.strip() else 'нет ответа' for t in texts]
    proba = _proba(safe)
    return float(np.mean(proba[:, SIMPLE_IDX]))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, PeftModel, prepare_model_for_kbit_training
from safetensors.torch import load_file as load_sf
from datasets import Dataset
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig, RewardTrainer, RewardConfig


bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

LORA_MODULES = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']

lora_causal = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=LORA_MODULES, bias='none',
    task_type=TaskType.CAUSAL_LM,
)
lora_cls = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=LORA_MODULES, bias='none',
    task_type=TaskType.SEQ_CLS,
)

def flush_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

def drop(*names):
    g = globals()
    for n in names:
        g.pop(n, None)
    flush_vram()

def adapter_ready(path):
    return os.path.isdir(path) and os.path.exists(f'{path}/adapter_config.json')

def latest_ckpt(path):
    if not os.path.isdir(path):
        return None
    cks = sorted(
        [d for d in os.listdir(path) if d.startswith('checkpoint-') and os.path.isdir(f'{path}/{d}')],
        key=lambda x: int(x.split('-')[-1])
    )
    return f'{path}/{cks[-1]}' if cks else None

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()
def strip_think(text):
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'<redacted_thinking>.*?</redacted_thinking>', '', text, flags=re.DOTALL)
    return text.strip()

def chat_text(tok, prompt, response=None):
    msgs = [{'role':'user','content':prompt}]
    if response is not None:
        msgs.append({'role':'assistant','content':response})
    gen_prompt = response is None
    try:
        return tok.apply_chat_template(msgs, tokenize=False,
                   add_generation_prompt=gen_prompt, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False,
                   add_generation_prompt=gen_prompt)

@torch.no_grad()
def generate_responses(model, tok, prompts, max_new=256):
    torch.manual_seed(SEED)
    model.eval()
    out = []
    for p in prompts:
        text = chat_text(tok, p)
        inp  = tok(text, return_tensors='pt', truncation=True, max_length=768).to(DEVICE)
        ids  = model.generate(**inp, max_new_tokens=max_new, do_sample=False,
                              pad_token_id=tok.eos_token_id)
        resp = tok.decode(ids[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
        out.append(strip_think(resp))
    return out

print('Configs ready.')


In [ ]:
class GoldRM(nn.Module):
    def __init__(self, lm, hidden_size):
        super().__init__()
        self.lm     = lm
        self.v_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask=None):
        out = self.lm(input_ids=input_ids, attention_mask=attention_mask,
                      output_hidden_states=True)
        h = out.hidden_states[-1]
        return self.v_head(h).squeeze(-1)

def load_gold_rm():
    tok = AutoTokenizer.from_pretrained(GOLD_RM_PATH, trust_remote_code=True)
    tok.pad_token = tok.pad_token or tok.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16,
        device_map='auto', trust_remote_code=True)
    lm = PeftModel.from_pretrained(base, GOLD_RM_PATH, is_trainable=False)
    lm.eval()

    hidden_size = base.config.hidden_size
    rm = GoldRM(lm, hidden_size).to(torch.bfloat16)

    vh = load_sf(f'{GOLD_RM_PATH}/value_head.safetensors')
    with torch.no_grad():
        for key, tensor in vh.items():
            t = tensor.to(torch.bfloat16)
            if 'weight' in key: rm.v_head.weight.copy_(t)
            elif 'bias' in key: rm.v_head.bias.copy_(t)

    rm.v_head = rm.v_head.to(DEVICE)
    rm.eval()
    print(f'Gold RM loaded. hidden_size={hidden_size}')
    return rm, tok

@torch.no_grad()
def score_with_rm(rm, tok, prompts, responses):
    rm.eval()
    scores = []
    for p, r in zip(prompts, responses):
        text = chat_text(tok, p, r)
        inp  = tok(text, return_tensors='pt', truncation=True, max_length=1024).to(DEVICE)
        vals = rm(**inp)
        last = inp['attention_mask'].sum() - 1
        scores.append(vals[0, last].item())
    return scores

print('Gold RM helpers ready.')


## Задача 1 — SFT на kid_adult
Обучаем модель отвечать простым языком (prompt → kid).  
Метрика: P_simple на public_test_style

In [ ]:
_tok_tmp = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
_tok_tmp.pad_token = _tok_tmp.pad_token or _tok_tmp.eos_token

def fmt_sft(ex):
    return {'text': chat_text(_tok_tmp, ex['prompt'], ex['kid'])}

sft_ds = Dataset.from_list(kid_adult_raw).map(
    fmt_sft, remove_columns=list(kid_adult_raw[0].keys()))
del _tok_tmp
print(f'SFT dataset: {len(sft_ds)} examples')
print('Sample:', sft_ds[0]['text'][:200])


In [ ]:
SFT_OUT = '/content/sft_adapter'
torch.manual_seed(SEED)

tok_sft   = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok_sft.pad_token = tok_sft.pad_token or tok_sft.eos_token
tok_sft.padding_side = 'right'

model_sft = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_cfg,
    device_map='auto', trust_remote_code=True)
model_sft = prepare_model_for_kbit_training(model_sft)

sft_cfg = SFTConfig(
    output_dir=SFT_OUT,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    max_length=512,
    dataset_text_field='text',
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    logging_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    seed=SEED, data_seed=SEED,
    report_to='none',
)

sft_trainer = SFTTrainer(
    model=model_sft,
    args=sft_cfg,
    train_dataset=sft_ds,
    processing_class=tok_sft,
    peft_config=lora_causal,
)
if adapter_ready(SFT_OUT):
    print(f'SFT adapter already saved at {SFT_OUT}, skipping training')
else:
    ckpt = latest_ckpt(SFT_OUT)
    if ckpt:
        print(f'Resuming SFT from {ckpt}')
    sft_trainer.train(resume_from_checkpoint=ckpt)
    sft_trainer.save_model(SFT_OUT)
    tok_sft.save_pretrained(SFT_OUT)
    print(f'SFT saved → {SFT_OUT}')
    drop('model_sft', 'sft_trainer', 'tok_sft')


In [ ]:
if not adapter_ready(SFT_OUT):
    raise RuntimeError('SFT adapter not found. Run the training cell above first.')

base1  = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
m1     = PeftModel.from_pretrained(base1, SFT_OUT, is_trainable=False)
tok1   = AutoTokenizer.from_pretrained(SFT_OUT, trust_remote_code=True)
tok1.pad_token = tok1.pad_token or tok1.eos_token

r1       = generate_responses(m1, tok1, TEST_STYLE_PROMPTS)
metric_1 = compute_p_simple(r1)
METRICS['task1_sft_p_simple'] = metric_1

print()
print('=' * 50)
print(f'METRIC 1  SFT P_simple = {metric_1:.4f}')
print('=' * 50)
drop('m1', 'base1', 'tok1')


## Задача 2 — DPO по стилю
chosen=kid, rejected=adult. Метрика: P_simple

In [ ]:
def fmt_dpo_style(ex):
    return {
        'prompt':   [{'role':'user',      'content': ex['prompt']}],
        'chosen':   [{'role':'assistant', 'content': ex['kid']}],
        'rejected': [{'role':'assistant', 'content': ex['adult']}],
    }

dpo_s_ds = Dataset.from_list(kid_adult_raw).map(
    fmt_dpo_style, remove_columns=list(kid_adult_raw[0].keys()))

DPO_S_OUT = '/content/dpo_style_adapter'
torch.manual_seed(SEED)

tok_s   = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok_s.pad_token = tok_s.pad_token or tok_s.eos_token
tok_s.padding_side = 'left'

model_s = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_cfg,
    device_map='auto', trust_remote_code=True)
model_s = prepare_model_for_kbit_training(model_s)

dpo_s_cfg = DPOConfig(
    output_dir=DPO_S_OUT,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    beta=0.1,
    max_length=512,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    logging_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    seed=SEED, report_to='none',
    remove_unused_columns=False,
)

dpo_s_trainer = DPOTrainer(
    model=model_s, ref_model=None,
    args=dpo_s_cfg,
    train_dataset=dpo_s_ds,
    processing_class=tok_s,
    peft_config=lora_causal,
)
if adapter_ready(DPO_S_OUT):
    print(f'DPO style adapter ready at {DPO_S_OUT}, skipping')
else:
    ckpt = latest_ckpt(DPO_S_OUT)
    dpo_s_trainer.train(resume_from_checkpoint=ckpt)
    dpo_s_trainer.save_model(DPO_S_OUT)
    tok_s.save_pretrained(DPO_S_OUT)
    print(f'DPO style saved → {DPO_S_OUT}')
    drop('model_s', 'dpo_s_trainer', 'tok_s')


In [ ]:
drop('model_s', 'dpo_s_trainer', 'tok_s')

base2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
m2    = PeftModel.from_pretrained(base2, DPO_S_OUT, is_trainable=False)
tok2  = AutoTokenizer.from_pretrained(DPO_S_OUT, trust_remote_code=True)
tok2.pad_token = tok2.pad_token or tok2.eos_token

r2       = generate_responses(m2, tok2, TEST_STYLE_PROMPTS)
metric_2 = compute_p_simple(r2)
METRICS['task2_dpo_style_p_simple'] = metric_2

print()
print('=' * 50)
print(f'METRIC 2  DPO-style P_simple = {metric_2:.4f}')
print('=' * 50)
drop('m2', 'base2', 'tok2')


## Задача 3 — Reward Model
Обучаем оценщика качества. Метрика: accuracy (chosen > rejected)

In [ ]:
from transformers import AutoModelForSequenceClassification

def fmt_rm(ex):
    return {
        'chosen':   [{'role':'user','content': ex['instruction']},
                     {'role':'assistant','content': ex['chosen']}],
        'rejected': [{'role':'user','content': ex['instruction']},
                     {'role':'assistant','content': ex['rejected']}],
    }

rm_ds = Dataset.from_list(good_bad_raw).map(
    fmt_rm, remove_columns=list(good_bad_raw[0].keys()))

RM_OUT = '/content/reward_model'
torch.manual_seed(SEED)

tok_rm   = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok_rm.pad_token = tok_rm.pad_token or tok_rm.eos_token
tok_rm.padding_side = 'right'

model_rm = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1,
    quantization_config=bnb_cfg,
    device_map='auto', trust_remote_code=True)
model_rm = prepare_model_for_kbit_training(model_rm)

rm_cfg = RewardConfig(
    output_dir=RM_OUT,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    max_length=512,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    logging_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    seed=SEED, report_to='none',
)

rm_trainer = RewardTrainer(
    model=model_rm,
    args=rm_cfg,
    train_dataset=rm_ds,
    processing_class=tok_rm,
    peft_config=lora_cls,
)
if adapter_ready(RM_OUT):
    print(f'RM adapter ready at {RM_OUT}, skipping')
else:
    ckpt = latest_ckpt(RM_OUT)
    rm_trainer.train(resume_from_checkpoint=ckpt)
    rm_trainer.save_model(RM_OUT)
    tok_rm.save_pretrained(RM_OUT)
    print(f'RM saved → {RM_OUT}')
    drop('model_rm', 'rm_trainer', 'tok_rm')


In [ ]:
drop('model_rm', 'rm_trainer', 'tok_rm')

trained_rm_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1,
    torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
trained_rm = PeftModel.from_pretrained(trained_rm_base, RM_OUT, is_trainable=False)
trained_rm.eval()
tok3 = AutoTokenizer.from_pretrained(RM_OUT, trust_remote_code=True)
tok3.pad_token = tok3.pad_token or tok3.eos_token

@torch.no_grad()
def score_seq_cls(rm, tok, prompts, responses):
    scores = []
    for p, r in zip(prompts, responses):
        text = chat_text(tok, p, r)
        inp  = tok(text, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
        out  = rm(**inp)
        scores.append(out.logits[0, 0].item())
    return scores

ch_sc  = score_seq_cls(trained_rm, tok3, TEST_QUALITY_PROMPTS, TEST_QUALITY_CHOSEN)
rej_sc = score_seq_cls(trained_rm, tok3, TEST_QUALITY_PROMPTS, TEST_QUALITY_REJECTED)

correct  = sum(c > r for c, r in zip(ch_sc, rej_sc))
metric_3 = correct / len(TEST_QUALITY_PROMPTS)
METRICS['task3_rm_accuracy'] = metric_3

print()
print('=' * 50)
print(f'METRIC 3  RM accuracy = {metric_3:.4f}  ({correct}/{len(TEST_QUALITY_PROMPTS)})')
print('=' * 50)
drop('trained_rm', 'trained_rm_base', 'tok3')


## Задача 4 — DPO по качеству
chosen=хорошее объяснение, rejected=плохое (оба простые). Метрика: gold_rm reward

In [ ]:
def fmt_dpo_q(ex):
    return {
        'prompt':   [{'role':'user',      'content': ex['instruction']}],
        'chosen':   [{'role':'assistant', 'content': ex['chosen']}],
        'rejected': [{'role':'assistant', 'content': ex['rejected']}],
    }

dpo_q_ds = Dataset.from_list(good_bad_raw).map(
    fmt_dpo_q, remove_columns=list(good_bad_raw[0].keys()))

DPO_Q_OUT = '/content/dpo_quality_adapter'
torch.manual_seed(SEED)

tok_q   = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok_q.pad_token = tok_q.pad_token or tok_q.eos_token
tok_q.padding_side = 'left'

model_q = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_cfg,
    device_map='auto', trust_remote_code=True)
model_q = prepare_model_for_kbit_training(model_q)

dpo_q_cfg = DPOConfig(
    output_dir=DPO_Q_OUT,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    beta=0.1,
    max_length=512,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    logging_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    seed=SEED, report_to='none',
    remove_unused_columns=False,
)

dpo_q_trainer = DPOTrainer(
    model=model_q, ref_model=None,
    args=dpo_q_cfg,
    train_dataset=dpo_q_ds,
    processing_class=tok_q,
    peft_config=lora_causal,
)
if adapter_ready(DPO_Q_OUT):
    print(f'DPO quality adapter ready at {DPO_Q_OUT}, skipping')
else:
    ckpt = latest_ckpt(DPO_Q_OUT)
    dpo_q_trainer.train(resume_from_checkpoint=ckpt)
    dpo_q_trainer.save_model(DPO_Q_OUT)
    tok_q.save_pretrained(DPO_Q_OUT)
    print(f'DPO quality saved → {DPO_Q_OUT}')
    drop('model_q', 'dpo_q_trainer', 'tok_q')


In [ ]:
drop('model_q', 'dpo_q_trainer', 'tok_q')

base4 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
m4    = PeftModel.from_pretrained(base4, DPO_Q_OUT, is_trainable=False)
tok4  = AutoTokenizer.from_pretrained(DPO_Q_OUT, trust_remote_code=True)
tok4.pad_token = tok4.pad_token or tok4.eos_token

r4 = generate_responses(m4, tok4, TEST_QUALITY_PROMPTS)
drop('m4', 'base4', 'tok4')

gold_rm, gold_rm_tok = load_gold_rm()
ref_scores = score_with_rm(gold_rm, gold_rm_tok, TEST_QUALITY_PROMPTS, TEST_QUALITY_REJECTED)
dpo_q_scores = score_with_rm(gold_rm, gold_rm_tok, TEST_QUALITY_PROMPTS, r4)

metric_4   = float(np.mean(dpo_q_scores))
win_rate_4 = float(np.mean([s > r for s, r in zip(dpo_q_scores, ref_scores)]))
METRICS['task4_dpo_quality_mean_reward'] = metric_4
METRICS['task4_dpo_quality_win_rate']    = win_rate_4

print()
print('=' * 50)
print(f'METRIC 4  DPO-quality mean gold_rm reward = {metric_4:.4f}')
print(f'          win rate vs rejected             = {win_rate_4:.4f}')
print('=' * 50)
drop('gold_rm', 'gold_rm_tok')


## Задача 5 — SimPO
Reference-free preference optimisation. Метрика: gold_rm reward

In [ ]:
try:
    from trl import CPOTrainer, CPOConfig
except ImportError:
    try:
        from trl.experimental.cpo import CPOTrainer, CPOConfig
    except ImportError:
        from trl.trainer.cpo_trainer import CPOTrainer
        from trl.trainer.cpo_config import CPOConfig

SIMPO_OUT = '/content/simpo_adapter'
torch.manual_seed(SEED)

tok_sp   = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok_sp.pad_token = tok_sp.pad_token or tok_sp.eos_token
tok_sp.padding_side = 'left'

model_sp = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_cfg,
    device_map='auto', trust_remote_code=True)
model_sp = prepare_model_for_kbit_training(model_sp)

simpo_cfg = CPOConfig(
    output_dir=SIMPO_OUT,
    loss_type='simpo',
    cpo_alpha=0.0,
    beta=2.0,
    simpo_gamma=0.5,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    max_length=512,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    logging_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    seed=SEED, report_to='none',
    remove_unused_columns=False,
)

simpo_trainer = CPOTrainer(
    model=model_sp,
    args=simpo_cfg,
    train_dataset=dpo_q_ds,   # same format as DPO quality
    processing_class=tok_sp,
    peft_config=lora_causal,
)
if adapter_ready(SIMPO_OUT):
    print(f'SimPO adapter ready at {SIMPO_OUT}, skipping')
else:
    ckpt = latest_ckpt(SIMPO_OUT)
    simpo_trainer.train(resume_from_checkpoint=ckpt)
    simpo_trainer.save_model(SIMPO_OUT)
    tok_sp.save_pretrained(SIMPO_OUT)
    print(f'SimPO saved → {SIMPO_OUT}')
    drop('model_sp', 'simpo_trainer', 'tok_sp')


In [ ]:
drop('model_sp', 'simpo_trainer', 'tok_sp')

base5 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
m5    = PeftModel.from_pretrained(base5, SIMPO_OUT, is_trainable=False)
tok5  = AutoTokenizer.from_pretrained(SIMPO_OUT, trust_remote_code=True)
tok5.pad_token = tok5.pad_token or tok5.eos_token

r5 = generate_responses(m5, tok5, TEST_QUALITY_PROMPTS)
drop('m5', 'base5', 'tok5')

gold_rm, gold_rm_tok = load_gold_rm()
ref_scores = score_with_rm(gold_rm, gold_rm_tok, TEST_QUALITY_PROMPTS, TEST_QUALITY_REJECTED)
simpo_scores = score_with_rm(gold_rm, gold_rm_tok, TEST_QUALITY_PROMPTS, r5)

metric_5   = float(np.mean(simpo_scores))
win_rate_5 = float(np.mean([s > r for s, r in zip(simpo_scores, ref_scores)]))
METRICS['task5_simpo_mean_reward'] = metric_5
METRICS['task5_simpo_win_rate']    = win_rate_5

print()
print('=' * 50)
print(f'METRIC 5  SimPO mean gold_rm reward = {metric_5:.4f}')
print(f'          win rate vs rejected       = {win_rate_5:.4f}')
print('=' * 50)
drop('gold_rm', 'gold_rm_tok')


## Итоговые метрики

In [ ]:
print('\n=== results ===')
for k, v in METRICS.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
